In [ ]:
!pip install transformers[torch] datasets scikit-learn evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.7 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import time
import torch
import numpy as np
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

print(f"GPU disponível? {'Sim' if torch.cuda.is_available() else 'Não'}")

df = pd.read_csv('dados_limpos_sem_stem.csv', sep=None, engine='python')

df.columns = df.columns.str.strip().str.lower()

if 'crime' in df.columns:
    df.rename(columns={'crime': 'label'}, inplace=True)

df['text'] = df['text'].astype(str)
df = df.dropna(subset=['text', 'label'])

df['label'] = df['label'].astype(int)

df_train_full, df_test = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
df_train, df_val = train_test_split(df_train_full, test_size=0.125, random_state=42, stratify=df_train_full['label'])

print(f"Tamanho do Treino: {len(df_train)}")
print(f"Tamanho da Validação: {len(df_val)}")
print(f"Tamanho do Teste: {len(df_test)}")

hg_dataset = DatasetDict({
    'train': Dataset.from_pandas(df_train),
    'val': Dataset.from_pandas(df_val),
    'test': Dataset.from_pandas(df_test)
})

model_name = "neuralmind/bert-base-portuguese-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

tokenized_datasets = hg_dataset.map(tokenize_function, batched=True)

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary', zero_division=0)
    acc = accuracy_score(labels, preds)
    return {'accuracy': acc, 'f1': f1, 'precision': precision, 'recall': recall}

training_args = TrainingArguments(
    output_dir="./resultados_bertimbau",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["val"],
    compute_metrics=compute_metrics,
)

print("\nIniciando fine-tuning do BERTimbau")
start_time_train = time.time()
trainer.train()
train_time = time.time() - start_time_train
print(f"\nTempo total de Treinamento: {train_time:.2f} segundos")

# avaliação final estritamente no conjunto de teste inedito
print("\nAvaliando o modelo na base de teste")
start_time_inf = time.time()
eval_results = trainer.evaluate(eval_dataset=tokenized_datasets["test"])
inf_time = time.time() - start_time_inf
print(f"Tempo de tnferência (teste): {inf_time:.4f} segundos")


trainer.save_model("./meu_bertimbau_crimes")
tokenizer.save_pretrained("./meu_bertimbau_crimes")
print("\nModelo treinado e salvo com sucesso")

GPU disponível? Sim
Tamanho do Treino: 7000
Tamanho da Validação: 1000
Tamanho do Teste: 2000


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/647 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/43.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/210k [00:00<?, ?B/s]

...


Tempo total de Treinamento: 498.90 segundos

Avaliando o modelo na base de teste


Training Loss,Validation Loss,Epoch,Accuracy,F1,Precision,Recall
0.231857,0.321061,3,0.882500,0.882910,0.879841,0.886000


Tempo de tnferência (teste): 14.8344 segundos


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Modelo treinado e salvo com sucesso


In [ ]:
# Compactar a pasta do modelo em um arquivo .zip
!zip -r meu_bertimbau_crimes.zip meu_bertimbau_crimes/

# Iniciar o download automático para a pasta "Downloads" do seu Windows/WSL
from google.colab import files
files.download('meu_bertimbau_crimes.zip')

	zip warning: name not matched: meu_bertimbau_crimes/

zip error: Nothing to do! (try: zip -r meu_bertimbau_crimes.zip . -i meu_bertimbau_crimes/)


FileNotFoundError: Cannot find file: meu_bertimbau_crimes.zip